# Vintage Alignment — All Components Leveled to 2022

Executes step 5 of `TRANSIT_DEMAND_PLAN.md`: brings every layer of the adjusted
all-mode matrix to a **2022** footing (the RavKav anchor year). Levels move, patterns
don't — the survey's destination structure is kept and only the margins are grown.

| Layer | Vintage | Treatment |
|---|---|---|
| CAR/OTHER base (THS survey) | 2018 | Furness: origin margin grown by population factors, destination margin by employment factors |
| Bus (RavKav × OnBoard) | 2022 | anchor — untouched |
| Train (smartcards) | 2019 | single national ridership factor 2019→2022 |

**Growth factors** come from the zonal socioeconomic files `Input/Zonal_2020.csv`
(observed) and `Input/Zonal_BU_2025.csv` (forecast): per area,
`g = (X_2025 / X_2020)^(4/5)` — the 2020→2025 annual rate applied over the four years
2018→2022 — computed separately for `POPULATION` (origin side) and `EMPL_TOT`
(destination side). Areas with (near-)zero residential population — pure employment
districts (Namal, Hutzot, Kiryat Nahum, Haifa Airport, Hamifrats, Matam) — use the
employment factor on the origin side too. May 2022 bus ridership was verified by the
study team as not COVID-suppressed, so no pandemic adjustment is applied to the base.

In [1]:
import numpy as np
import pandas as pd

POP_MIN = 500          # below this 2020 population, an area's origin factor falls back to employment

z20 = pd.read_csv('Input/Zonal_2020.csv', encoding='utf-8', encoding_errors='replace')
z25 = pd.read_csv('Input/Zonal_BU_2025.csv', encoding='utf-8', encoding_errors='replace')
sub = pd.read_excel('Input/Submatrix_tazs.xlsx')
taz_to_area = sub.set_index('TAZ')['AggAreaCode']
legend = sub.drop_duplicates('AggAreaCode').set_index('AggAreaCode')['AggAreaName']

agg = {}
for tag, z in (('20', z20), ('25', z25)):
    a = z.groupby(z['TAZ_ID'].map(taz_to_area))[['POPULATION', 'EMPL_TOT']].sum()
    agg['p' + tag], agg['e' + tag] = a['POPULATION'], a['EMPL_TOT']
fac = pd.DataFrame(agg)
fac['g_pop'] = (fac['p25'] / fac['p20']) ** (4 / 5)
fac['g_emp'] = (fac['e25'] / fac['e20']) ** (4 / 5)
fac['g_orig'] = np.where(fac['p20'] >= POP_MIN, fac['g_pop'], fac['g_emp'])
fac['g_dest'] = fac['g_emp']
fac.insert(0, 'AggAreaName', legend.reindex(fac.index))
fac.index.name = 'AggAreaCode'
fac.round(4).to_csv('Output/transit/area_growth_factors_2018_2022.csv')
print(fac[['AggAreaName', 'p20', 'p25', 'g_pop', 'g_emp', 'g_orig']].round(3).to_string())

                       AggAreaName     p20     p25  g_pop  g_emp  g_orig
AggAreaCode                                                             
1.0                    TiratCarmel   24681   32809  1.256  1.153   1.256
2.0                          Matam     390     405  1.031  1.034   1.034
3.0                     Neot Peres    4211    4243  1.006  1.059   1.006
4.0                     Neve David   14578   14622  1.002  0.986   1.002
5.0                      Ein Hayam    6021    6272  1.033  1.020   1.033
6.0                      Bat Galim    4729    4800  1.012  1.008   1.012
7.0                 Kiryat Eliezer   14558   15187  1.034  1.024   1.034
8.0                      Hamoshava    4842    5388  1.089  1.018   1.089
9.0                     Lower City    2651    2995  1.103  1.022   1.103
10.0                  Hadar Carmel    7482    8471  1.104  1.064   1.104
11.0                         Namal       0       0    NaN  1.018   1.018
12.0                    Neve Yosef    3519    3658 

In [2]:
def load_area(name, areas=None):
    m = pd.read_csv(name, index_col=0)
    m.index = m.index.astype(int)
    m.columns = m.columns.astype(int)
    if areas is not None:
        m = m.reindex(index=areas, columns=areas, fill_value=0)
    return m

bus_area = load_area('Output/bus/bus_od_area_new_filtered.csv')
AREAS = list(bus_area.index)                     # the 25 noise-filtered areas
train_area = load_area('Output/train/train_od_area.csv', AREAS)
ths_all = load_area('Output/ths2017/study_taz/submatrices/matrix_avg_ALL_area.csv', AREAS)
ths_transit = load_area('Output/ths2017/study_taz/submatrices/matrix_avg_TRANSIT_area.csv', AREAS)
ths_rail = load_area('Output/ths2017/study_taz/submatrices/matrix_avg_RAIL_area.csv', AREAS)

base18 = ths_all - ths_transit - ths_rail        # survey CAR + OTHER, 2018
g_o = fac['g_orig'].reindex(AREAS)
g_d = fac['g_dest'].reindex(AREAS)
assert g_o.notna().all() and g_d.notna().all()
print(f"CAR/OTHER base 2018: {base18.values.sum():,.0f} trips over {len(AREAS)} areas")
print(f"origin factors: min {g_o.min():.3f} ({fac.loc[g_o.idxmin(), 'AggAreaName']}), "
      f"max {g_o.max():.3f} ({fac.loc[g_o.idxmax(), 'AggAreaName']}), "
      f"trip-weighted mean {np.average(g_o, weights=base18.sum(axis=1)):.3f}")
print(f"destination factors: min {g_d.min():.3f}, max {g_d.max():.3f}, "
      f"trip-weighted mean {np.average(g_d, weights=base18.sum(axis=0)):.3f}")

CAR/OTHER base 2018: 235,899 trips over 25 areas
origin factors: min 0.948 (Nesher Lower), max 1.256 (TiratCarmel), trip-weighted mean 1.067
destination factors: min 0.975, max 1.164, trip-weighted mean 1.064


## Furness the CAR/OTHER base to 2022 margins

Row targets = 2018 origin totals × population factor (AM-peak origins are
predominantly homes); column targets = 2018 destination totals × employment factor
(AM-peak destinations are predominantly work/school), rescaled so both margins share
the origin-side grand total. The 2018 matrix is the seed, so the destination structure
moves only as much as the margins force it to.

In [3]:
def furness(seed, row_t, col_t, iters=200, tol=1e-8):
    m = seed.values.astype(float).copy()
    rt, ct = row_t.values.astype(float), col_t.values.astype(float)
    for _ in range(iters):
        rs = m.sum(axis=1)
        m *= np.divide(rt, rs, out=np.zeros_like(rt), where=rs > 0)[:, None]
        cs = m.sum(axis=0)
        m *= np.divide(ct, cs, out=np.ones_like(ct), where=cs > 0)[None, :]
        if max(np.abs(m.sum(axis=1) - rt).max(), np.abs(m.sum(axis=0) - ct).max()) < tol:
            break
    return pd.DataFrame(m, index=seed.index, columns=seed.columns)

row_t = base18.sum(axis=1) * g_o
col_t = base18.sum(axis=0) * g_d
col_t *= row_t.sum() / col_t.sum()               # one grand total, set by the origin side

base22 = furness(base18, row_t, col_t)
base22.index.name = 'AggAreaCode'
base22.to_csv('Output/transit/car_other_area_2022.csv', float_format='%.6g')
chg = base22.values.sum() / base18.values.sum() - 1
print(f"CAR/OTHER 2022: {base22.values.sum():,.0f} trips ({chg:+.1%} vs 2018)")
print(f"margin fit: max row-target error {np.abs(base22.sum(axis=1) - row_t).max():.2e}, "
      f"max col-target error {np.abs(base22.sum(axis=0) - col_t).max():.2e}")

CAR/OTHER 2022: 251,684 trips (+6.7% vs 2018)
margin fit: max row-target error 2.77e-05, max col-target error 3.64e-12


## Train 2019 → 2022 and the leveled all-mode matrix

National heavy-rail ridership: 69M passengers (2019) → 54.7M (2022), factor **0.793**.
Unlike bus (verified back to routine by May 2022), rail recovery lagged nationally, so
the 2019 smartcard matrix is scaled down to its 2022 level. Applied as a single scalar
to the small in-sub-area train layer.

In [4]:
RAIL_FACTOR_2019_2022 = 54.7 / 69.0

train22 = train_area * RAIL_FACTOR_2019_2022
all22 = base22 + bus_area + train22
all22.index.name = 'AggAreaCode'
all22.to_csv('Output/transit/all_adjusted_area_2022.csv', float_format='%.6g')

leg_full = pd.read_csv('Output/ths2017/study_taz/submatrices/area_legend.csv').set_index('AggAreaCode')
share = pd.DataFrame({
    'AggAreaName': fac['AggAreaName'].reindex(AREAS).values,
    'IsLRT_Corridor': leg_full['IsLRT_Corridor'].reindex(AREAS).values,
    'car_other': base22.sum(axis=1).round(0),
    'bus': bus_area.sum(axis=1).round(0),
    'train': train22.sum(axis=1).round(0),
}, index=AREAS)
share['total'] = share[['car_other', 'bus', 'train']].sum(axis=1)
share['transit_share'] = ((share['bus'] + share['train']) / share['total'].replace(0, np.nan)).round(3)
share.index.name = 'AggAreaCode'
share.to_csv('Output/transit/mode_share_area_2022.csv')

corr = share[share['IsLRT_Corridor'] == 1]
print(f"ALL_adjusted 2022: {all22.values.sum():,.0f} trips "
      f"(CAR/OTHER {base22.values.sum():,.0f} + bus {bus_area.values.sum():,.0f} "
      f"+ train {train22.values.sum():,.0f})")
print(f"train scaled 2019->2022 by {RAIL_FACTOR_2019_2022:.3f}: "
      f"{train_area.values.sum():,.0f} -> {train22.values.sum():,.0f}")
print(f"transit share, 2022 base: all areas "
      f"{(share['bus'].sum() + share['train'].sum()) / share['total'].sum():.1%} | corridor "
      f"{(corr['bus'].sum() + corr['train'].sum()) / corr['total'].sum():.1%}")

m18 = pd.read_csv('Output/transit/mode_share_area.csv', index_col=0)
comp = pd.DataFrame({'AggAreaName': share['AggAreaName'],
                     'share_mixed_vintage': m18['transit_share'].reindex(AREAS),
                     'share_2022base': share['transit_share']})
comp['delta'] = (comp['share_2022base'] - comp['share_mixed_vintage']).round(3)
print("\nlargest share changes vs the mixed-vintage table:")
print(comp.reindex(comp['delta'].abs().sort_values(ascending=False).index).head(8).to_string())

ALL_adjusted 2022: 276,355 trips (CAR/OTHER 251,684 + bus 23,909 + train 763)
train scaled 2019->2022 by 0.793: 962 -> 763
transit share, 2022 base: all areas 8.9% | corridor 9.9%

largest share changes vs the mixed-vintage table:
                       AggAreaName  share_mixed_vintage  share_2022base  delta
AggAreaCode                                                                   
13                       Hamifrats                0.495           0.481 -0.014
9                       Lower City                0.131           0.119 -0.012
1                      TiratCarmel                0.063           0.051 -0.012
2                            Matam                0.305           0.294 -0.011
15                    Kiryat Nahum                0.144           0.134 -0.010
10                    Hadar Carmel                0.096           0.088 -0.008
25           Kiryat Motzkin-Bialik                0.086           0.079 -0.007
8                        Hamoshava                0.095   

## Notes

- Only **volumes** were leveled; the hybrid probabilities, correction factors and the
  OnBoard destination pattern are untouched (patterns age slowly; vintage correction
  belongs on margins).
- The factor `(X_2025/X_2020)^(4/5)` assumes each area's 2020→2025 trend also held over
  2018→2022 — grounded in the observed 2020 zonal data and the official 2025 forecast.
- The 2022 footing still mixes frames: CAR/OTHER is residents-2018 grown demographically,
  the transit layer is everyone-2022; the frame caveat from `TRANSIT_DEMAND_PLAN.md`
  stands (non-residents appear in the transit layer only).
- Per the study team's checks, May 2022 bus ridership was **not** COVID-suppressed —
  no pandemic correction is applied anywhere except the rail layer, where the national
  ridership series itself (54.7M vs 69M) shows 2022 below the 2019 vintage of the
  smartcard matrix.